# PadelVision — 01 Player analytics (Kaggle / Colab)

Runs phase 1 on a full match: player detection + tracking → 4 player identities → distance,
speed, sprints, zones, team formation and heatmaps. Then renders a short annotated preview to check tracking by eye.

**Needs `court.json`.** Make it on your laptop with `padelvision calibrate data/court_frame.png`,
then either add it to the same Kaggle Dataset as the video, or paste its contents in the cell below.

**Kaggle:** Accelerator **GPU T4 x2**, **Internet on**, video attached as a Dataset.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    print("Videos found:", *videos, sep="
  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
# court.json: pasted below, else ml/calibrations/<video name>.json in the repo, else court.json next to the video.
COURT_JSON_TEXT = """
"""

court_files = sorted(Path("/kaggle/input" if ON_KAGGLE else "/content/drive/MyDrive/padel").rglob("court.json"))
# Calibrations committed to the repo, named after the video file (e.g. ml/calibrations/Test_video.json).
repo_cal = Path(REPO) / "ml" / "calibrations" / f"{Path(VIDEO_PATH).stem}.json"
if repo_cal.exists():
    court_files.insert(0, repo_cal)
COURT_JSON = f"{WORK}/court.json"
if COURT_JSON_TEXT.strip():
    Path(COURT_JSON).write_text(COURT_JSON_TEXT)
elif court_files:
    Path(COURT_JSON).write_text(court_files[0].read_text())
    print("Using", court_files[0])
else:
    raise FileNotFoundError("No court.json: attach it to the dataset or paste it above")

from padelvision.court import CourtCalibration
cal = CourtCalibration.load(COURT_JSON)
print(f"court error {cal.reprojection_error_m()*100:.0f} cm, lens k1={cal.lens.k1:.3f} k2={cal.lens.k2:.3f}")

In [ ]:
RUN_DIR = f"{WORK}/runs/match1"
STRIDE = 1          # 2 = analyse every other frame (2x faster)
MAX_FRAMES = None   # e.g. 900 for a quick 30 s test at 30 fps

extra = f"--max-frames {MAX_FRAMES}" if MAX_FRAMES else ""
!cd {REPO} && python -m padelvision.cli analyze "{VIDEO_PATH}" --court {COURT_JSON} --out {RUN_DIR} --stride {STRIDE} {extra}

**Check the court alignment:** yellow lines should sit on the painted lines (red dots = your clicks).

In [ ]:
from IPython.display import Image, display
display(Image(f"{RUN_DIR}/court_overlay.png"))
display(Image(f"{RUN_DIR}/heatmaps.png"))

In [ ]:
import json
import pandas as pd

stats = json.loads(Path(f"{RUN_DIR}/stats.json").read_text())
rows = []
for p, s in stats["players"].items():
    zones = s.get("zones", {})
    rows.append({
        "player": p, "team": s["team"], "tracked": f"{s['tracked_fraction']:.0%}",
        "distance_m": s.get("distance_m"), "avg_speed_m/s": s.get("avg_speed_mps"),
        "peak_speed_m/s": s.get("peak_speed_mps"), "sprints": s.get("sprints"),
        "net%": zones.get("net"), "back%": zones.get("back"),
        "note": s.get("insufficient_data", ""),
    })
display(pd.DataFrame(rows).set_index("player"))
print(json.dumps(stats["teams"], indent=2))

In [ ]:
# 30 s annotated preview (boxes, player ids, court lines, mini court), converted to H.264 for the browser.
PREVIEW_START_S = 0  # must be inside the analysed part (see MAX_FRAMES)
!cd {REPO} && python -m padelvision.cli render "{VIDEO_PATH}" --run {RUN_DIR} --start {PREVIEW_START_S} --seconds 30 --out {RUN_DIR}/preview_raw.mp4
!ffmpeg -y -loglevel error -i {RUN_DIR}/preview_raw.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 28 {RUN_DIR}/preview.mp4 && rm {RUN_DIR}/preview_raw.mp4

from IPython.display import Video
Video(f"{RUN_DIR}/preview.mp4", embed=True, width=960)

**Send back for review:** File → Download Notebook (includes all outputs).
Also download `runs/match1/` from the Output panel (at least `detections.parquet`, `video.json`, `stats.json`):
with those, identity and stats logic can be re-tuned locally via `padelvision restats` without the GPU.
